In [1]:
# System
import os
import sys

os.environ["KERAS_BACKEND"] = "jax"
sys.path.append("../..")

In [2]:
# Setup
import json
from importlib import import_module
from pathlib import Path

import numpy as np
from keras import ops
from rich.table import Table

from src.models import GradientBoostedDecisionTree as BDT
from src.models import LearnableCutFlowParallel as LCF_PAR
from src.models import LearnableCutFlowSequential as LCF_SEQ
from src.models import MultiLayerPerceptron as MLP
from src.utils import Timer, load_model, print, to_numpy

In [3]:
# Parameters
rerun = False

# Dataset
dataset = "mock5"  # *
n_samples = 200000
seed = 42

# Model
centers = [-2, 2, 0, 0, -1, -5, -1.8]  # *
features = [r"$x_1$", r"$x_2$", r"$x_3$", r"$x_4$", r"$x_5$", r"$x_7$", r"$x_9$"]  # *

n_epochs = 200
batch_size = 512
fit_verbose = 2
predict_verbose = 0

if not rerun and Path("results.json").exists():
    with open("results.json", "r") as f:
        results = json.load(f)
else:
    results = {}

In [4]:
# Dataset
module = import_module(f"src.datasets.{dataset}")
load_data = getattr(module, "load_data")
(x_train, y_train), (x_test, y_test) = load_data(n_samples, seed)

print(f"{x_train.shape=}")
print(f"{y_train.shape=}")
print(f"{x_test.shape=}")
print(f"{y_test.shape=}")

x_train.shape=(100000, 7)
y_train.shape=(100000, 1)
x_test.shape=(100000, 7)
y_test.shape=(100000, 1)


In [5]:
# Model: BDT
bdt_name = "bdt"
bdt_ckpt_path = Path(f"checkpoints/{bdt_name}.pkl")

if rerun or not bdt_ckpt_path.exists():
    bdt = BDT(input_shape=x_train.shape, name=bdt_name)
    bdt.compile(optimizer="adam", loss="crossentropy")

    with Timer() as timer:
        bdt.fit(
            x_train,
            y_train.squeeze(),
            batch_size=batch_size,
            epochs=n_epochs,
            verbose=fit_verbose,
        )
    training_time = timer.record
    results[bdt_name] = {"training_time": training_time}

    bdt.save(bdt_ckpt_path)
    bdt_ckpt = load_model(bdt_ckpt_path)
else:
    training_time = results[bdt_name]["training_time"]

print(f"Training time: {training_time:.2f} seconds")

Training time: 29.80 seconds


In [6]:
# Model: MLP
mlp_name = "mlp"
mlp_ckpt_path = Path(f"checkpoints/{mlp_name}.keras")

if rerun or not mlp_ckpt_path.exists():
    mlp = MLP(x_train.shape, name=mlp_name)
    mlp.adapt(x_train)
    mlp.compile(optimizer="adam", loss="crossentropy")

    with Timer() as timer:
        mlp.fit(
            x_train,
            y_train,
            batch_size=batch_size,
            epochs=n_epochs,
            verbose=fit_verbose,
        )
    training_time = timer.record
    results[mlp_name] = {"training_time": training_time}

    mlp.save(mlp_ckpt_path)
    mlp_ckpt = load_model(mlp_ckpt_path)
else:
    training_time = results[mlp_name]["training_time"]

print(f"Training time: {training_time:.2f} seconds")

Training time: 47.05 seconds


In [7]:
# Model: LCF(parallel)
lcf_par_name = "lcf_par"
lcf_par_ckpt_path = Path(f"checkpoints/{lcf_par_name}.keras")

if rerun or not lcf_par_ckpt_path.exists():
    lcf_par = LCF_PAR(x_train.shape, centers, features=features, name=lcf_par_name)
    lcf_par.adapt(x_train)
    lcf_par.compile(optimizer="adam", loss="crossentropy")

    with Timer() as timer:
        lcf_par.fit(
            x_train,
            y_train,
            batch_size=batch_size,
            epochs=n_epochs,
            verbose=fit_verbose,
        )
    training_time = timer.record
    results[lcf_par_name] = {"training_time": training_time}

    lcf_par.save(lcf_par_ckpt_path)
    lcf_par_ckpt = load_model(lcf_par_ckpt_path)
else:
    training_time = results[lcf_par_name]["training_time"]

print(f"Training time: {training_time:.2f} seconds")

Training time: 51.71 seconds


In [8]:
# Model: LCF(sequential)
lcf_seq_name = "lcf_seq"
lcf_seq_ckpt_path = Path(f"checkpoints/{lcf_seq_name}.keras")

if rerun or not lcf_seq_ckpt_path.exists():
    lcf_seq = LCF_SEQ(x_train.shape, centers, features=features, name=lcf_seq_name)
    lcf_seq.adapt(x_train)
    lcf_seq.compile(optimizer="adam", loss="crossentropy")

    with Timer() as timer:
        lcf_seq.fit(
            x_train,
            y_train,
            batch_size=batch_size,
            epochs=n_epochs,
            verbose=fit_verbose,
        )
    training_time = timer.record
    results[lcf_seq_name] = {"training_time": training_time}

    lcf_seq.save(lcf_seq_ckpt_path)
    lcf_seq_ckpt = load_model(lcf_seq_ckpt_path)
else:
    training_time = results[lcf_seq_name]["training_time"]

print(f"Training time: {training_time:.2f} seconds")

Training time: 54.50 seconds


In [9]:
# Analysis: metrics
y_true = y_test

table = Table(title="Model Performance Comparison")
table.add_column("#", justify="center", style="cyan", no_wrap=True)
table.add_column("Model", style="magenta")
table.add_column("TP", justify="right", style="green")
table.add_column("FP", justify="right", style="red")
table.add_column("Accuracy", justify="right", style="blue")
table.add_column("Precision", justify="right", style="blue")
table.add_column("Significance", justify="right", style="yellow")
table.add_column("Time(s)", justify="right", style="yellow")

if rerun or not Path("results.json").exists():
    for i, ckpt in enumerate([bdt_ckpt, mlp_ckpt, lcf_par_ckpt, lcf_seq_ckpt]):
        y_pred = ckpt.predict(x_test, batch_size=batch_size, verbose=0)
        y_pred = ops.all(y_pred > 0.5, axis=1, keepdims=True)

        tp = to_numpy(ops.sum((y_true == 1) & (y_pred == 1)))
        fp = to_numpy(ops.sum((y_true == 0) & (y_pred == 1)))
        tn = to_numpy(ops.sum((y_true == 0) & (y_pred == 0)))
        fn = to_numpy(ops.sum((y_true == 1) & (y_pred == 0)))

        accuracy = (tp + tn) / (tp + tn + fp + fn)
        precision = tp / (tp + fp)

        s = tp / (tp + tn + fp + fn) * 3000 * 1000 * 0.7644
        b = fp / (tp + tn + fp + fn) * 3000 * 1000 * 1.806 * 1e5
        significance = s / np.sqrt(b)

        results[ckpt.name].update(
            {
                "tp": tp.tolist(),
                "fp": fp.tolist(),
                "accuracy": accuracy.tolist(),
                "precision": precision.tolist(),
                "significance": significance.tolist(),
            }
        )

        with open("results.json", "w") as f:
            json.dump(results, f, indent=4)

for i, (name, metrics) in enumerate(results.items()):
    tp = metrics["tp"]
    fp = metrics["fp"]
    accuracy = metrics["accuracy"]
    precision = metrics["precision"]
    significance = metrics["significance"]
    training_time = metrics["training_time"]

    table.add_row(
        str(i + 1),
        name,
        f"{tp:.0f}",
        f"{fp:.0f}",
        f"{accuracy:.4f}",
        f"{precision:.4f}",
        f"{significance:.4f}",
        f"{training_time:.2f}",
    )

print(table)

                         Model Performance Comparison                         
┏━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ # ┃ Model   ┃    TP ┃   FP ┃ Accuracy ┃ Precision ┃ Significance ┃ Time(s) ┃
┡━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ 1 │ bdt     │ 48860 │  997 │   0.9801 │    0.9800 │      15.2450 │   29.80 │
│ 2 │ mlp     │ 48876 │ 1065 │   0.9795 │    0.9787 │      14.7551 │   47.05 │
│ 3 │ lcf_par │ 16542 │    6 │   0.6668 │    0.9996 │      66.5327 │   51.71 │
│ 4 │ lcf_seq │ 39742 │ 1586 │   0.8830 │    0.9616 │       9.8315 │   54.50 │
└───┴─────────┴───────┴──────┴──────────┴───────────┴──────────────┴─────────┘
